# Experiment 3 — Secondary Dataset Evaluation (Reddit Mental Health Posts)

This notebook evaluates Qwen2.5 7B Instruct on a secondary dataset derived from Reddit mental health posts.

The dataset is sampled, manually labelled, and analysed using the same structured prompt pipeline from previous experiments. The goal is to assess model performance under more realistic and noisy language conditions.


## 1. Environment Setup

This section installs the required libraries for model loading, inference, and evaluation.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas openpyxl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00


## 2. Hugging Face Authentication

A Hugging Face token is required to access the selected open-weight model. The token is entered securely at runtime.

In [ ]:
from getpass import getpass
HF_TOKEN = getpass('Enter your Hugging Face token: ')

Enter your Hugging Face token: ··········


## 3. Core Functions and Evaluation Setup

This section defines the labels, helper functions, and evaluation methods used in the experiment.

It includes:
- Label normalisation  
- Output extraction logic  
- Rule-based fallback classification  
- Evaluation metrics  

These functions ensure robust label detection and consistent evaluation across the dataset.

In [ ]:

import pandas as pd
import torch
import gc
import json
import re
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LABELS = ["safe", "indicator", "ideation", "behavior", "attempt"]

def normalize_label(label):
    if pd.isna(label):
        return None
    label = str(label).strip().lower()
    mapping = {"behaviour": "behavior"}
    return mapping.get(label, label)

def extract_json_label(text):
    if not text:
        return None, None, text

    import json
    import re

    text_lower = text.lower()

    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            label = normalize_label(obj.get("label"))
            reason = obj.get("reason", "")
            if label in LABELS:
                return label, reason, text
        except:
            pass

    if any(phrase in text_lower for phrase in [
        "want to die", "end my life", "kill myself",
        "don't want to live", "tired of living", "ready to die"
    ]):
        return "ideation", None, text

    if any(phrase in text_lower for phrase in [
        "plan to", "going to do it", "tonight", "method"
    ]):
        return "behavior", None, text

    if any(phrase in text_lower for phrase in [
        "attempt", "already did", "took pills", "cut myself"
    ]):
        return "attempt", None, text

    if any(phrase in text_lower for phrase in [
        "sad", "depressed", "alone", "tired", "hopeless"
    ]):
        return "indicator", None, text

    return "safe", None, text

## 4. Dataset Upload, Extraction, and Inspection

The secondary dataset is uploaded as a compressed ZIP file and extracted within the notebook environment.

After extraction, the CSV file is loaded and inspected to verify the dataset shape, column names, and sample records. This confirms that the correct text field is available for model evaluation.

Because this dataset is derived from Reddit posts, it contains informal and unstructured language, making it more challenging than the controlled dialogue dataset used in previous experiments.

In [ ]:
from google.colab import files
import pandas as pd
import zipfile
import io
import os

uploaded = files.upload()
filename = list(uploaded.keys())[0]

with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
    zip_ref.extractall()

print("Files inside ZIP:", os.listdir())

df2 = pd.read_csv("reddit_depression_suicidewatch.csv", encoding="latin-1")

print("Shape:", df2.shape)
print("Columns:", df2.columns.tolist())
df2.head()

Saving archive.zip to archive (2).zip
Files inside ZIP: ['.config', 'archive (1).zip', 'archive.zip', 'reddit_depression_suicidewatch.csv', 'archive (2).zip', 'sample_data']
Shape: (20363, 2)
Columns: ['text', 'label']


,text,label
0,I recently went through a breakup and she said...,depression
1,"I do not know how to navigate these feelings, ...",depression
2,"So I have been with my bf for 5 months , and h...",depression
3,I am so exhausted of this. Just when I think I...,SuicideWatch
4,I have been severly bullied since i was 5 till...,depression


## 5. Text Column Selection

The `text` column is selected as the input field for model evaluation. This column contains the raw Reddit post content used for classification.



In [ ]:

TEXT_COL = "text"

df2[[TEXT_COL]].head(10)


,text
0,I recently went through a breakup and she said...
1,"I do not know how to navigate these feelings, ..."
2,"So I have been with my bf for 5 months , and h..."
3,I am so exhausted of this. Just when I think I...
4,I have been severly bullied since i was 5 till...
5,I am 20 year old with some good friends but I ...
6,My mom made me go to a camp that she knows I h...
7,Help me for ideas simple healthy meals to make...
8,it is looming around the corner again. It alwa...
9,there is.....foodAnd other things I will be ju...


## 6. Sampling and Dataset Preparation

A subset of 10 samples is randomly selected from the secondary dataset to create a manageable evaluation set.

Each sample is assigned a unique Case ID (R1–R10) for tracking predictions. The text column is renamed to "Paraphrased Dialogue" to maintain consistency with previous experiments.

This step ensures that the dataset structure aligns with the evaluation pipeline used earlier.

In [ ]:

df2_sample = df2[[TEXT_COL]].sample(10, random_state=42).reset_index(drop=True).copy()
df2_sample["Case ID"] = [f"R{i+1}" for i in range(len(df2_sample))]
df2_sample.rename(columns={TEXT_COL: "Paraphrased Dialogue"}, inplace=True)

display(df2_sample)


,Paraphrased Dialogue,Case ID
0,I was going to do it last night but I knew som...,R1
1,Literally right after I posted how I always fu...,R2
2,Recently a lot has been going on in my life. I...,R3
3,There is no long commentary hereJust know ever...,R4
4,I have called my crisis team but have to wait ...,R5
5,My son you were suffering from the disease #Fi...,R6
6,TW: SHThrowaway for obvious reasons. No matter...,R7
7,"My Iife pretty much fell apart today, I have a...",R8
8,"I am E. My mum is J[08/07, 19:08] E: that is i...",R9
9,"This might be sloppy because of mobile, very s...",R10


The sampled dataset includes diverse and informal expressions of emotional distress, reflecting real-world variability in language. This makes the classification task more challenging compared to the controlled dataset used in Experiment 1.

## 7. Manual Annotation and Ground Truth Labels

The sampled Reddit posts are manually annotated using the five predefined suicide risk categories:

- safe  
- indicator  
- ideation  
- behavior  
- attempt  

Each post is assigned a single label based on its level of expressed emotional distress or suicidal intent.

This manual annotation process establishes the ground truth labels required for evaluating model performance.

In [ ]:

manual_labels = [
    "ideation",
    "ideation",
    "indicator",
    "indicator",
    "behavior",
    "indicator",
    "ideation",
    "ideation",
    "indicator",
    "indicator"
]
assert len(manual_labels) == len(df2_sample), "Number of labels must match number of sampled rows."
df2_sample["true_label"] = [normalize_label(x) for x in manual_labels]

display(df2_sample[["Case ID", "Paraphrased Dialogue", "true_label"]])


,Case ID,Paraphrased Dialogue,true_label
0,R1,I was going to do it last night but I knew som...,ideation
1,R2,Literally right after I posted how I always fu...,ideation
2,R3,Recently a lot has been going on in my life. I...,indicator
3,R4,There is no long commentary hereJust know ever...,indicator
4,R5,I have called my crisis team but have to wait ...,behavior
5,R6,My son you were suffering from the disease #Fi...,indicator
6,R7,TW: SHThrowaway for obvious reasons. No matter...,ideation
7,R8,"My Iife pretty much fell apart today, I have a...",ideation
8,R9,"I am E. My mum is J[08/07, 19:08] E: that is i...",indicator
9,R10,"This might be sloppy because of mobile, very s...",indicator


The labelled dataset reflects varying levels of emotional expression, including indirect and ambiguous language.

Manual labelling is necessary for this dataset, as real-world data does not contain predefined risk categories. This introduces subjectivity but provides a realistic evaluation scenario for model performance.

## 8. Model Configuration — Qwen2.5 7B Instruct

The Qwen2.5 7B Instruct model is loaded using 4-bit quantisation to reduce GPU memory usage while maintaining acceptable performance.

This allows the model to run efficiently within limited computational resources.

A reusable loading function is defined to initialise both the tokenizer and model consistently.

In [ ]:

QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HF_TOKEN,
        trust_remote_code=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    return tokenizer, model

qwen_tokenizer, qwen_model = load_model(QWEN_MODEL)
print("Qwen loaded.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen loaded.


The model is now ready for inference and will be applied to the labelled dataset for evaluation.

## 9. Prompt Engineering and Inference Pipeline

A few-shot prompting strategy is used to guide the model toward producing accurate and structured classifications.

The prompt includes example dialogues with corresponding labels and explanations, allowing the model to learn expected patterns before classifying new inputs.

A strict JSON output format is enforced to ensure consistency and enable reliable extraction of predicted labels.

In [ ]:

FEW_SHOT_PROMPT = """
You are a strict suicide risk classification assistant.

Classify each dialogue into exactly one label:
safe, indicator, ideation, behavior, attempt

Return valid JSON only:
{"label": "<one label>", "reason": "<brief explanation>"}

Example 1:
Dialogue:
User: I had a rough day, but I know things will get better.
AI: I'm glad you're holding on. Is there anything helping you cope right now?
Output:
{"label": "safe", "reason": "The user expresses distress but no suicide-related warning signs."}

Example 2:
Dialogue:
User: Sometimes I feel like disappearing and not waking up.
AI: That sounds really painful. Have these thoughts become more frequent recently?
Output:
{"label": "ideation", "reason": "The dialogue suggests suicidal thoughts without clear evidence of action."}

Now classify the next dialogue.
""".strip()

def build_prompt(dialogue: str) -> str:
    return f"""{FEW_SHOT_PROMPT}

Dialogue:
{dialogue}
""".strip()


The classification function constructs the prompt, performs model inference, decodes the output, and extracts the predicted label.

This pipeline ensures that raw model outputs are transformed into structured predictions suitable for evaluation.

In [ ]:

def classify_dialogue_local(tokenizer, model, dialogue: str, max_new_tokens: int = 120):
    prompt = build_prompt(dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text

    label, reason, raw = extract_json_label(generated_text)
    return {
        "label": label,
        "reason": reason,
        "raw_output": raw
    }


## 10. Model Evaluation — Qwen2.5 7B Instruct

The model is applied to each dialogue case in the secondary dataset.

For each input, the predicted label, reasoning, and raw output are recorded. These predictions are compared against the manually assigned ground truth labels to evaluate performance.

In [ ]:
results2 = []

for _, row in df2_sample.iterrows():
    out = classify_dialogue_local(
        qwen_tokenizer,
        qwen_model,
        row["Paraphrased Dialogue"]
    )

    results2.append({
        "Case ID": row["Case ID"],
        "true_label": row["true_label"],
        "pred_label": out["label"],
        "reason": out["reason"],
        "raw_output": out["raw_output"]
    })

    print(f"Done: {row['Case ID']} -> {out['label']}")
    print("RAW:", out["raw_output"])
    print("-" * 80)

results2_df = pd.DataFrame(results2)
results2_df.head(10)

Done: R1 -> ideation
RAW: . I am tired of living. I am done. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I am ready. I
--------------------------------------------------------------------------------
Done: R2 -> attempt
RAW: .
{"label": "attempt", "reason": "The user explicitly states their intention to end their life and provides a timeline indicating immediate plans."} {"label": "attempt", "reason": "The user explicitly states their intention to end their life and provides a timeline indicating immediate plans."}
--------------------------------------------------------------------------------
Done: R3 -> safe
RAW: I do not want to live anymore. I do not want to wake up. I do not want to be here. I do not want to exist. I

,Case ID,true_label,pred_label,reason,raw_output
0,R1,ideation,ideation,None,. I am tired of living. I am done. I am ready....
1,R2,ideation,attempt,The user explicitly states their intention to ...,".\n{""label"": ""attempt"", ""reason"": ""The user ex..."
2,R3,indicator,safe,None,I do not want to live anymore. I do not want t...
3,R4,indicator,ideation,The dialogue contains strong expressions of ho...,I am a failure. I am a failure. I am a failure...
4,R5,behavior,ideation,The user describes intense emotional pain and ...,".\n{""label"": ""ideation"", ""reason"": ""The user d..."
5,R6,indicator,safe,The dialogue discusses a medical condition wit...,AI: I'm sorry to hear about your son's conditi...
6,R7,ideation,attempt,The dialogue explicitly mentions plans to end ...,"{""label"": ""attempt"", ""reason"": ""The dialogue e..."
7,R8,ideation,behavior,The user expresses a plan to act on their suic...,"I don't want to do it alone.\n{""label"": ""behav..."
8,R9,indicator,indicator,None,she is not here for me now? She has never care...
9,R10,indicator,ideation,The dialogue shows clear suicidal ideation wit...,I am so tired. I am so tired. I am so tired.\n...


## 11. Evaluation Results and Observations

The results indicate that model performance on the secondary dataset is lower compared to the controlled dataset used in Experiment 1.

This is expected due to:
- Informal and unstructured language in Reddit posts  
- Ambiguity in emotional expression  
- Lack of clear contextual cues  

Several misclassifications occur between closely related categories such as *indicator* and *ideation*, highlighting the difficulty of interpreting nuanced real-world text.

Despite this, the model is still able to capture strong signals of distress in more explicit cases, demonstrating partial robustness.

## 11. Quantitative Evaluation

The model predictions are evaluated against the manually assigned ground truth labels using standard classification metrics.

These include:
- Accuracy  
- Macro Precision  
- Macro Recall  
- Macro F1-score  

A classification report and confusion matrix are also generated to provide detailed insights into model performance across different risk categories.

In [ ]:

def evaluate_model(y_true, y_pred, model_name):
    valid_pairs = [(t, p) for t, p in zip(y_true, y_pred) if p in LABELS]
    y_true_clean = [t for t, p in valid_pairs]
    y_pred_clean = [p for t, p in valid_pairs]

    acc = accuracy_score(y_true_clean, y_pred_clean)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_clean,
        y_pred_clean,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
    cm = confusion_matrix(y_true_clean, y_pred_clean, labels=LABELS)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro Precision: {precision:.3f}")
    print(f"Macro Recall: {recall:.3f}")
    print(f"Macro F1: {f1:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_true_clean, y_pred_clean, labels=LABELS, zero_division=0))
    print("Confusion Matrix:")
    print(cm)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    }, cm

summary2, cm2 = evaluate_model(
    results2_df["true_label"].tolist(),
    results2_df["pred_label"].tolist(),
    "Qwen2.5 7B Instruct (Reddit Dataset)"
)

pd.DataFrame([summary2])



=== Qwen2.5 7B Instruct (Reddit Dataset) ===
Accuracy: 0.200
Macro Precision: 0.250
Macro Recall: 0.090
Macro F1: 0.117

Classification Report:
              precision    recall  f1-score   support

        safe       0.00      0.00      0.00         0
   indicator       1.00      0.20      0.33         5
    ideation       0.25      0.25      0.25         4
    behavior       0.00      0.00      0.00         1
     attempt       0.00      0.00      0.00         0

    accuracy                           0.20        10
   macro avg       0.25      0.09      0.12        10
weighted avg       0.60      0.20      0.27        10

Confusion Matrix:
[[0 0 0 0 0]
 [2 1 2 0 0]
 [0 0 1 1 2]
 [0 0 1 0 0]
 [0 0 0 0 0]]


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Qwen2.5 7B Instruct (Reddit Dataset),0.2,0.25,0.09,0.116667


## 12. Quantitative Results Analysis

The model achieves relatively low performance on the secondary dataset, with an accuracy of 0.20 and a macro F1-score of 0.117.

This indicates that the model struggles to generalise to real-world conversational data, particularly when expressions of distress are indirect or ambiguous.

The confusion matrix shows that most errors occur between closely related categories such as *indicator* and *ideation*, suggesting difficulty in distinguishing subtle differences in risk levels.

Additionally, the model fails to correctly identify higher-risk categories such as *behavior* and *attempt*, highlighting limitations in detecting more critical cases within noisy data.

Overall, these results reinforce the importance of dataset quality, prompt design, and model robustness when applying LLMs to real-world mental health tasks.

## 13. Exporting Results

The prediction results and summary metrics are saved as CSV files for inclusion in the report and appendix.

This allows for reproducibility and enables further analysis outside the notebook environment.

In [ ]:

results2_df.to_csv("dataset2_results.csv", index=False)
pd.DataFrame([summary2]).to_csv("dataset2_summary.csv", index=False)

print("Saved: dataset2_results.csv")
print("Saved: dataset2_summary.csv")


Saved: dataset2_results.csv
Saved: dataset2_summary.csv


## 14. Resource Cleanup

To ensure efficient resource usage, the model and tokenizer are removed from memory after evaluation.

This step helps prevent GPU memory issues when running multiple experiments.

In [ ]:

del qwen_model
del qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")


## Final Summary

This experiment demonstrates that model performance significantly decreases when applied to real-world, unstructured data compared to controlled datasets.

This highlights the importance of dataset quality, prompt design, and robust evaluation when developing AI systems for sensitive applications such as mental health.